# Template Run — BLIP-2 Fusion VQA

**Notebook nay chi dung de CHAY script, khong viet logic tai day.**

Thu tu:
1. Chon GPU: Runtime > Change runtime type > T4 hoac A100
2. Chay Cell 1 (Mount Drive + Clone repo)
3. Chay Cell 2 (Cai dependencies)
4. Sua bien o Cell 3 (duong dan, ten EXP, run_name)
5. Chay Cell 4 (Pre-extract features — bo qua neu da co cache)
6. Chay Cell 5 (Train)
7. Chay Cell 6 (Evaluate)
8. Chay Cell 7 (Resume — dung khi Colab bi disconnect)

## Cell 1 — Mount Drive & Clone repo

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Clone repo lan dau
import os
REPO_DIR = "/content/blip2-fusion-experiment-vqa"
GITHUB_USER = "<username>"   # <-- SUA THANH USERNAME GITHUB CUA BAN

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/{GITHUB_USER}/blip2-fusion-experiment-vqa.git {REPO_DIR}
    print("Clone hoan thanh.")
else:
    # Pull ban moi nhat neu repo da ton tai
    !git -C {REPO_DIR} pull
    print("Pull hoan thanh.")

%cd {REPO_DIR}
!git log --oneline -5

## Cell 2 — Cai dependencies & Dang nhap W&B

In [ ]:
!pip install -r requirements.txt -q

import torch
print("PyTorch :", torch.__version__)
print("CUDA    :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU     :", torch.cuda.get_device_name(0))
    print("VRAM    :", round(torch.cuda.get_device_properties(0).total_memory/1e9,1), "GB")

# Dang nhap W&B (dung API key tu wandb.ai/authorize)
import wandb
wandb.login()

## Cell 3 — Cau hinh (SUA O DAY TRUOC KHI CHAY)

In [ ]:
# ================================================================
# CHINH SUA CAC BIEN NAY CHO TUNG LAN CHAY
# ================================================================

EXP_ID      = "01"           # 01 | 02 | 03 | 04 | 05 | 06 | 07
RUN_NUMBER  = "1"            # lan chay thu may (1, 2, 3...)
YOUR_NAME   = "ten"          # ten viet tat cua ban (khoa, minh, tuan...)

DATA_ROOT   = "/content/drive/MyDrive/blip2_project/data"
ANSWER_LIST = f"{DATA_ROOT}/ans2idx.json"

# ================================================================
# Cac bien duoc tao tu dong — KHONG can sua
# ================================================================
CONFIG_FILE = f"configs/exp{EXP_ID}.yaml"
OUTPUT_DIR  = f"{DATA_ROOT}/checkpoints/exp{EXP_ID}"
RUN_NAME    = f"exp{EXP_ID}_lan{RUN_NUMBER}_{YOUR_NAME}"

print(f"Config      : {CONFIG_FILE}")
print(f"Output dir  : {OUTPUT_DIR}")
print(f"W&B run name: {RUN_NAME}")
print(f"Data root   : {DATA_ROOT}")
print(f"Answer list : {ANSWER_LIST}")

## Cell 4 — Pre-extract CLIP features (chi chay 1 lan dau, bo qua neu da co cache)

In [ ]:
import os, h5py

CACHE_DIR   = f"{DATA_ROOT}/cache"
train_h5    = f"{CACHE_DIR}/train_features.h5"
val_h5      = f"{CACHE_DIR}/val_features.h5"

if os.path.exists(train_h5) and os.path.exists(val_h5):
    with h5py.File(train_h5, "r") as f:
        print(f"Cache da co: train = {len(f.keys()):,} anh")
    with h5py.File(val_h5, "r") as f:
        print(f"Cache da co: val   = {len(f.keys()):,} anh")
    print("Bo qua pre-extract.")
else:
    print("Chua co cache — bat dau pre-extract (~15-20 phut tren T4)...")
    !python data/pre_extract_features.py \
        --split       both \
        --data_root   "{DATA_ROOT}" \
        --output_dir  "{CACHE_DIR}" \
        --vqav2_dir   vqav2 \
        --batch_size  64 \
        --ckpt_every  10
    print("Pre-extract hoan thanh.")

## Cell 5 — Train

In [ ]:
!python scripts/train.py \
    --config      "{CONFIG_FILE}" \
    --run_name    "{RUN_NAME}" \
    --data_root   "{DATA_ROOT}" \
    --vqav2_dir   vqav2 \
    --coco_dir    coco \
    --cache_dir   cache \
    --answer_list "{ANSWER_LIST}" \
    --output_dir  "{OUTPUT_DIR}"

## Cell 6 — Evaluate (chay sau khi train xong)

In [ ]:
CHECKPOINT  = f"{OUTPUT_DIR}/best_model.pth"   # hoac checkpoint_epoch_XXX.pth
EVAL_OUTPUT = f"{OUTPUT_DIR}/eval_results.json"

!python scripts/evaluate.py \
    --config     "{CONFIG_FILE}" \
    --checkpoint "{CHECKPOINT}" \
    --split      val \
    --output     "{EVAL_OUTPUT}"

# In ket qua ra man hinh
import json
if __import__('os').path.exists(EVAL_OUTPUT):
    with open(EVAL_OUTPUT) as f:
        results = json.load(f)
    print("\n" + "="*45)
    print(f"KET QUA DANH GIA — {RUN_NAME}")
    print("="*45)
    for k, v in results.items():
        print(f"  {k:<25}: {v}")
    print("="*45)
    print("\nGhi vao bang theo doi:")
    print(f"  Run name : {RUN_NAME}")
    overall = results.get('overall_accuracy', results.get('val_acc', '?'))
    print(f"  Val Acc  : {overall}")

## Cell 7 — Resume (Colab bi disconnect — chay lai tu day)

In [ ]:
# Sau khi Colab restart, chay Cell 1 + 2 + 3 truoc, sau do chay cell nay.
# QUAN TRONG: giu nguyen RUN_NAME giong lan dau de W&B tiep tuc cung 1 run.

!python scripts/train.py \
    --config      "{CONFIG_FILE}" \
    --run_name    "{RUN_NAME}" \
    --data_root   "{DATA_ROOT}" \
    --vqav2_dir   vqav2 \
    --coco_dir    coco \
    --cache_dir   cache \
    --answer_list "{ANSWER_LIST}" \
    --output_dir  "{OUTPUT_DIR}" \
    --resume      auto

## Cell 8 — Giu Colab khoi timeout (tuy chon)

Dan doan JavaScript duoi vao **Console cua trinh duyet** (F12 -> Console) de giu ket noi:

```javascript
function ClickConnect(){
  console.log('Giu ket noi Colab...');
  document.querySelector('#top-toolbar > colab-connect-button')
    .shadowRoot.querySelector('#connect').click();
}
setInterval(ClickConnect, 60000);
```